In [ ]:
#imports and installation
!pip install torch transformers peft bitsandbytes accelerate datasets

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training, TaskType
from datasets import load_dataset
import bitsandbytes as bnb
import numpy as np

In [ ]:
#loading & processing dataset

ds = load_dataset("BuildaByte/Meditation-miniset-v0.2")["train"]

#getting unique prompts (must split by unique prompts or else duplicates exist)

unique_prompts = list(set(ds["user_prompt"]))
np.random.seed(42)
np.random.shuffle(unique_prompts)

n = len(unique_prompts)
train_prompts = set(unique_prompts[: int(n * 0.80)])
val_prompts = set(unique_prompts[int(n * 0.80): int(n * 0.90)])
test_prompts = set(unique_prompts[int(n * 0.90):])

#splitting rows based on which bucket their prompt falls into

train_ds = ds.filter(lambda x: x["user_prompt"] in train_prompts)
val_ds = ds.filter(lambda x: x["user_prompt"] in val_prompts)
test_ds = ds.filter(lambda x: x["user_prompt"] in test_prompts)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
#loading base model and QLoRA setup
model_name = "microsoft/Phi-3-mini-4k-instruct"

print(f"Starting to load the model {model_name}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, #enables 4-bit quantization for the model (QLoRA)
    device_map="auto"
)

#important to turn off use_fast for better reliability (+ slowness is actually useful for this case)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

print(f"Successfully loaded the model {model_name}!")

In [ ]:
#defining LoRA configuration

lora_config = LoraConfig(
    #rank and target_modules expanded due to stylistic nature of guided meditation
    r=16,                      
    lora_alpha=32,           
    lora_dropout=0.05,         
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   #full attention block
        "gate_proj", "up_proj", "down_proj",      #MLP layers
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,  
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
#preparing the model for fine-tuning
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [ ]:
#tokenizing and preparing the dataset for training

from tokenizer import generate_and_tokenize_prompt

tokenized_train_dataset = train_ds.map(generate_and_tokenize_prompt)
tokenized_val_dataset = val_ds.map(generate_and_tokenize_prompt)
tokenized_test_dataset = test_ds.map(generate_and_tokenize_prompt)

#testing tokenization
print("User prompt: " + test_ds[1]['user_prompt'])
print("Meditation guidance: " + test_ds[1]['meditation_guidance'] + "\n")

In [ ]:
#evaluating the model on the test set before training
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_prompt = "I want to meditate on the concept of impermanence. Please guide me through a meditation session that helps me understand and accept the transient nature of life."

model_input = tokenizer(eval_prompt, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=256, pad_token_id=2)[0], skip_special_tokens=True))

In [ ]:
#fine-tuning the model with QLoRA

project = "OmBot-Meditation-LLM"
base_model_name = "phi-3-mini-4k-instruct"
run_name = base_model_name + "-" + project
output_dir = "./" + run_name

tokenizer.pad_token = tokenizer.eos_token #requires attention mask

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_ratio=0.03,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=6000, #high overhead for best performance
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        load_best_model_at_end=True,
        learning_rate=2e-4, 
        logging_steps=50,
        fp16=True, #change to bf16 if using better GPU
        optim="paged_adamw_8bit", #dramatically reduces memory usage and speeds up training
        logging_dir="./logs",        
        save_strategy="steps", #save the model checkpoint every logging step
        save_steps=250, #save checkpoints every 250 steps
        evaluation_strategy="steps", #evaluate the model every logging step
        eval_steps=250, #evaluate and save checkpoints every 250 steps
        do_eval=True,            
        report_to='none',           
        run_name=f"{run_name}-{datetime.now().strftime('%Y-%m-%d-%H-%M')}",          # Name of the W&B run (optional)
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
model.config.use_cache = False  #re-enable for inference
trainer.train()

In [ ]:
#plotting training results (loss curve)
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_loss = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_loss = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.plot(*zip(*train_loss), label="train loss")
plt.plot(*zip(*eval_loss), label="eval loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

In [ ]:
#inference testing after training
from tokenizer import build_prompt 

model.eval()

sample = {
    "user_experience_level": "beginner",
    "context": "feeling anxious before a work presentation",
    "user_prompt": "I want to meditate on the concept of impermanence.",
    "suggested_techniques": "breathing exercises",
    "meditation_style": "guided visualization",
}
prompt_only, _ = build_prompt(sample)

inputs = tokenizer(prompt_only, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
#evaluating the fine-tuned model on the held-out test set
#free-form generation -> use ROUGE-L overlap against the reference guidance rather than exact-match accuracy
!pip install rouge-score

from rouge_score import rouge_scorer
from tokenizer import build_prompt

#ROUGE-L = longest-common-subsequence overlap between generated and reference text
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

model.eval()
rouge_l_scores = []
samples_to_show = []

for row in test_ds:
    prompt_only, _ = build_prompt(row)
    inputs = tokenizer(prompt_only, return_tensors="pt", truncation=True, max_length=400).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    reference = row["meditation_guidance"]

    score = scorer.score(reference, generated)["rougeL"].fmeasure
    rouge_l_scores.append(score)

    if len(samples_to_show) < 3:
        samples_to_show.append((row["user_prompt"], reference, generated))

rouge_l_scores = np.array(rouge_l_scores)
print(f"Test set size: {len(rouge_l_scores)}")
print(f"ROUGE-L F1: mean={rouge_l_scores.mean():.3f}, min={rouge_l_scores.min():.3f}, max={rouge_l_scores.max():.3f}")

for prompt, reference, generated in samples_to_show:
    print(f"\n--- Prompt: {prompt}")
    print(f"Reference: {reference}")
    print(f"Generated: {generated}")
